 Verificar a integridade dos dados do pipeline
 vamos verificar as camadas do pipeline se foram realizadas de forma correta 
 As camadas são :
 Controle
 Qualidade
 Trusted
 Refined



# Importaçao das Bibliotecas

In [1]:
import duckdb
import pathlib
import pandas as pd
from pathlib import Path
import pandas as pd
from pathlib import Path
from pyspark.sql.functions import input_file_name, split, col, lit, regexp_replace, element_at
from datetime import datetime, timedelta

# Caminho dos Arquivos

In [2]:
# Base do Data Lake
LAKE_ROOT = Path(r"C:\Data_Lake_PoD_Cartoes\datalake")

# Camadas de Controle e Qualidade (Salvas diretamente na raiz do datalake)
controle = LAKE_ROOT / "controle"
qualidade = LAKE_ROOT / "qualidade"

# Camada Trusted (gravado do pipeline)
trusted_fatura = LAKE_ROOT / "trusted" / "tb_01_fatura"
trusted_pagamentos = LAKE_ROOT / "trusted" / "tb_02_pagamento"

# Camada Refined (Stage e Book de Variáveis)
refined_stage_fatura = LAKE_ROOT / "refined" / "stage_fatura"
refined_book_fatura = LAKE_ROOT / "refined" / "book_fatura"   

# Iniciar Spark

In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Analise_DataLake").getOrCreate()

c:\Data_Lake_PoD_Cartoes\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


# Verificação dos dados

In [4]:
# Visualizar Log de Controle (Auditoria / Lineage)

df_controle = spark.read.parquet(str(controle))
df_controle.show(truncate=False)


+---------------+--------------+-------------+------------------------------------------+-----------------------------------------+--------------------------+
|tabela         |dt_proc       |qtd_registros|camada_origem                             |arquivo_origem                           |executado_em              |
+---------------+--------------+-------------+------------------------------------------+-----------------------------------------+--------------------------+
|tb_02_pagamento|20260728201448|7709         |001_raw/pagamento                         |tb_pagamentos_20240901_20240902000000.csv|2026-07-28 20:15:19.166726|
|book_fatura    |20260728201543|15553        |002_trusted/tb_01_fatura + tb_02_pagamento|safras_consolidadas                      |2026-07-28 20:17:15.985918|
|tb_01_fatura   |20260728201354|15553        |001_raw/fatura                            |tb_fatura_20240901_20240902000000.csv    |2026-07-28 20:14:24.956475|
+---------------+--------------+-------------+

In [5]:
# Visualizar Log de Qualidade (Integridade)

df_qualidade = spark.read.parquet(str(qualidade))
df_qualidade.show(truncate=False)

+---------------+--------------+------+------+---------------+-----------------+
|tabela         |dt_proc       |status|falhas|total_registros|chaves_duplicadas|
+---------------+--------------+------+------+---------------+-----------------+
|tb_02_pagamento|20260728201448|OK    |      |7709           |0                |
|tb_01_fatura   |20260728201354|OK    |      |15553          |0                |
|book_fatura    |20260728201543|OK    |      |15553          |0                |
+---------------+--------------+------+------+---------------+-----------------+



In [6]:
# Visualizar Faturas na Trusted

df_trusted_fatura = spark.read.parquet(str(trusted_fatura))
df_trusted_fatura.show(5)

+--------------+----------+---------+------------+---------------+------------+----------------------+------+
|       dt_proc|id_cliente|id_fatura|data_emissao|data_vencimento|valor_fatura|valor_pagamento_minimo|   ref|
+--------------+----------+---------+------------+---------------+------------+----------------------+------+
|20260728201354|         1|        3|  2023-03-01|     2023-03-06|     3265.83|                  0.00|202303|
|20260728201354|         2|       14|  2023-03-01|     2023-03-06|     2785.94|                  0.00|202303|
|20260728201354|         3|       28|  2023-03-01|     2023-03-06|     2287.70|                  0.00|202303|
|20260728201354|         4|       41|  2023-03-01|     2023-03-06|      176.09|                  0.00|202303|
|20260728201354|         5|       57|  2023-03-01|     2023-03-06|     4463.19|                  0.00|202303|
+--------------+----------+---------+------------+---------------+------------+----------------------+------+
only showi

In [7]:
# Visualizar Pagamentos na Trusted

df_trusted_pagamentos = spark.read.parquet(str(trusted_pagamentos))
df_trusted_pagamentos.show(5)

+--------------+----------+---------+------------+--------------+---------------+------+
|       dt_proc|id_cliente|id_fatura|id_pagamento|data_pagamento|valor_pagamento|   ref|
+--------------+----------+---------+------------+--------------+---------------+------+
|20260728201448|         1|        7|           5|    2023-07-11|        1374.19|202307|
|20260728201448|        10|      142|          56|    2023-07-16|        2722.62|202307|
|20260728201448|        13|      189|          79|    2023-07-20|        3324.65|202307|
|20260728201448|        14|      206|          89|    2023-07-21|        4195.49|202307|
|20260728201448|        15|      222|          95|    2023-07-18|        1024.65|202307|
+--------------+----------+---------+------------+--------------+---------------+------+
only showing top 5 rows


In [9]:
# Visualizar o Book de Variáveis (Refined)

df_book = spark.read.parquet(str(refined_book_fatura))
df_book.show(10)

+--------------+----------+-----------------------------------+-------------------------------------+-------------------------------------+-------------------------------------+-------------------------------------+------------------------------------+------------------------------------+------------------------------------+---------------------------------+-----------------------------------+-----------------------------------+-----------------------------------+-----------------------------------+----------------------------------+----------------------------------+----------------------------------+---------------------------------+-----------------------------------+-----------------------------------+-----------------------------------+-----------------------------------+----------------------------------+----------------------------------+----------------------------------+---------------------------------+-----------------------------------+-----------------------------------+--